# RealMLP


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics import roc_auc_score
from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"

train_df = pl.read_parquet(processed_data_dir / "train_with_folds.parquet")
test_df = pl.read_parquet(processed_data_dir / "test_clean.parquet")

train_df.shape, test_df.shape

((699635, 25), (299844, 23))

In [5]:
target = train_df["satisfaction"]
fold = train_df["fold"]
train_ids = train_df["id"]
test_ids = test_df["id"]

features = train_df.drop("id", "satisfaction", "fold")
test_features = test_df.drop("id")

features.shape, test_features.shape

((699635, 22), (299844, 22))

In [9]:
features = train_df.drop("id", "satisfaction", "fold").to_pandas()
test_features = test_df.drop("id").to_pandas()

features["Class"] = features["Class"].astype("category")
test_features["Class"] = test_features["Class"].astype("category")

target = train_df["satisfaction"].to_numpy()
fold = train_df["fold"].to_numpy()

features.dtypes

Age                                     int64
Flight Distance                         int64
Inflight wifi service                   int64
Departure/Arrival time convenient       int64
Ease of Online booking                  int64
Gate location                           int64
Food and drink                          int64
Online boarding                         int64
Seat comfort                            int64
Inflight entertainment                  int64
On-board service                        int64
Leg room service                        int64
Baggage handling                        int64
Checkin service                         int64
Cleanliness                             int64
Departure Delay in Minutes              int64
Arrival Delay in Minutes              float64
Gender                                   int8
Customer Type                            int8
Type of Travel                           int8
Class                                category
Arrival Delay Missing             

In [10]:
valid_fold = 0

train_mask = fold != valid_fold
valid_mask = fold == valid_fold

x_train = features.loc[train_mask].reset_index(drop=True)
x_valid = features.loc[valid_mask].reset_index(drop=True)

y_train = target[train_mask]
y_valid = target[valid_mask]

x_train.shape, x_valid.shape, y_train.shape, y_valid.shape

((559708, 22), (139927, 22), (559708,), (139927,))

In [11]:
x_test = test_features.copy()

arrival_median = x_train["Arrival Delay in Minutes"].median()

x_train["Arrival Delay in Minutes"] = (
    x_train["Arrival Delay in Minutes"].fillna(arrival_median)
)

x_valid["Arrival Delay in Minutes"] = (
    x_valid["Arrival Delay in Minutes"].fillna(arrival_median)
)

x_test["Arrival Delay in Minutes"] = (
    x_test["Arrival Delay in Minutes"].fillna(arrival_median)
)

x_train.isna().sum().sum(), x_valid.isna().sum().sum(), x_test.isna().sum().sum()

(np.int64(0), np.int64(0), np.int64(0))

In [12]:
realmlp_model = RealMLP_TD_Classifier(
    device="cpu",
    random_state=42,
    n_epochs=20,
    val_metric_name="1-auc_ovr",
    n_cv=1,
    n_refit=0,
    verbosity=2,
)

In [13]:
%%time

realmlp_model.fit(
    x_train,
    y_train,
    X_val=x_valid,
    y_val=y_valid,
)

Columns classified as continuous: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender', 'Customer Type', 'Type of Travel', 'Arrival Delay Missing']
Columns classified as categorical: ['Class']


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 1/20: val 1-auc_ovr = 0.045179
Epoch 2/20: val 1-auc_ovr = 0.045995
Epoch 3/20: val 1-auc_ovr = 0.044335
Epoch 4/20: val 1-auc_ovr = 0.042796
Epoch 5/20: val 1-auc_ovr = 0.044294
Epoch 6/20: val 1-auc_ovr = 0.045613
Epoch 7/20: val 1-auc_ovr = 0.045272
Epoch 8/20: val 1-auc_ovr = 0.044064
Epoch 9/20: val 1-auc_ovr = 0.042584
Epoch 10/20: val 1-auc_ovr = 0.042512
Epoch 11/20: val 1-auc_ovr = 0.043918
Epoch 12/20: val 1-auc_ovr = 0.045222
Epoch 13/20: val 1-auc_ovr = 0.044678
Epoch 14/20: val 1-auc_ovr = 0.046431
Epoch 15/20: val 1-auc_ovr = 0.044554
Epoch 16/20: val 1-auc_ovr = 0.044161
Epoch 17/20: val 1-auc_ovr = 0.043283
Epoch 18/20: val 1-auc_ovr = 0.042500
Epoch 19/20: val 1-auc_ovr = 0.041998
Epoch 20/20: val 1-auc_ovr = 0.041808


`Trainer.fit` stopped: `max_epochs=20` reached.


CPU times: total: 1h 30min 37s
Wall time: 27min 29s


,device,'cpu'
,random_state,42
,verbosity,2
,val_metric_name,'1-auc_ovr'
,n_epochs,20
,n_cv,1
,n_refit,0
,n_repeats,1
,val_fraction,0.2
,n_threads,None
,tmp_folder,None


In [14]:
valid_prediction = realmlp_model.predict_proba(x_valid)[:, 1]

valid_auc = roc_auc_score(y_valid, valid_prediction)

valid_auc

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


0.9581919060553339

In [15]:
oof_dir = project_root / "artifacts" / "oof"

realmlp_oof = np.full(len(features), np.nan)
realmlp_oof[valid_mask] = valid_prediction

np.save(oof_dir / "realmlp_partial.npy", realmlp_oof)

In [16]:
np.isnan(realmlp_oof).sum(), np.isnan(realmlp_oof).mean()

(np.int64(559708), np.float64(0.8))

In [17]:
test_prediction = realmlp_model.predict_proba(x_test)[:, 1]

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


In [19]:
test_dir = project_root / "artifacts" / "test_predictions"

np.save(
    test_dir / "realmlp_fold_0.npy",
    test_prediction,
)
test_prediction.shape, np.isnan(test_prediction).sum()

((299844,), np.int64(0))

In [20]:
xgboost_oof = np.load(oof_dir / "xgboost_oof.npy")
lightgbm_oof = np.load(oof_dir / "lightgbm.npy")
catboost_oof = np.load(oof_dir / "catboost.npy")

In [21]:
fold_scores = pd.Series({
    "xgboost": roc_auc_score(y_valid, xgboost_oof[valid_mask]),
    "lightgbm": roc_auc_score(y_valid, lightgbm_oof[valid_mask]),
    "catboost": roc_auc_score(y_valid, catboost_oof[valid_mask]),
    "realmlp": roc_auc_score(y_valid, valid_prediction),
})

fold_scores.sort_values(ascending=False)

lightgbm    0.959056
xgboost     0.958885
catboost    0.958800
realmlp     0.958192
dtype: float64

In [2]:
import torch

torch.backends.mps.is_available()

False